# AQUA20 — Marine Species Classification with ResNet50

Transfer learning on ImageNet-pretrained ResNet50 for 20-class marine species classification using the [AQUA20 dataset](https://huggingface.co/datasets/taufiktrf/AQUA20).

**Pipeline:**
1. Load AQUA20 (8,171 images, 20 species) with train/val/test splits
2. Build ResNet50 with ImageNet weights + new randomly initialized 20-class head
3. **Stage 1** — Freeze backbone, train head only (25 epochs)
4. **Stage 2** — Unfreeze backbone, fine-tune end-to-end (75 epochs)
5. Evaluate: Top-1 / Top-3 accuracy, macro F1, confusion matrix

---
**Resume & persistence:** If `best_model.pth` exists, Stage 1 is skipped automatically.
All outputs are copied to `output/` for easy download.
On Kaggle, save a notebook version to persist outputs across sessions.

In [ ]:
!pip install torch>=2.0.0 torchvision>=0.15.0 datasets>=2.14.0 scikit-learn>=1.2.0 matplotlib>=3.7.0 seaborn>=0.12.0 tqdm>=4.65.0 numpy>=1.24.0 Pillow>=9.5.0 -q

---
## 1. Imports & Configuration

In [ ]:
import os
import json
import shutil
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datasets import load_dataset, load_dataset_builder
from tqdm import tqdm
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# ── Config ──────────────────────────────────────────────
NUM_CLASSES = 20
BATCH_SIZE = 32
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 75
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "best_model.pth"
STATE_PATH = "training_state.json"
OUTPUT_DIR = "output"
NUM_WORKERS = os.cpu_count()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Num workers: {NUM_WORKERS}")

---
## 2. Data Module — Load AQUA20 & Create DataLoaders

In [ ]:
def get_transform(train=True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(IMG_SIZE + 32),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])


def transform_images(examples, transform_fn):
    examples["image"] = [transform_fn(img.convert("RGB")) for img in examples["image"]]
    return examples


def get_dataloaders():
    dataset = load_dataset("AQUA20")
    split = dataset["train"].train_test_split(test_size=0.2, seed=42)
    train_split = split["train"]
    val_split = split["test"]
    test_split = dataset["test"]

    train_split.set_transform(
        lambda ex: transform_images(ex, get_transform(train=True))
    )
    val_split.set_transform(
        lambda ex: transform_images(ex, get_transform(train=False))
    )
    test_split.set_transform(
        lambda ex: transform_images(ex, get_transform(train=False))
    )

    train_loader = DataLoader(
        train_split, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
    )
    val_loader = DataLoader(
        val_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    test_loader = DataLoader(
        test_split, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    return train_loader, val_loader, test_loader


print("Data module ready.")

---
## 3. Model — Pretrained ResNet50 with New 20-Class Head

In [ ]:
def build_model(num_classes=NUM_CLASSES):
    model = torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    nn.init.kaiming_normal_(model.fc.weight, mode="fan_out", nonlinearity="relu")
    nn.init.zeros_(model.fc.bias)
    return model


def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True


print("Model module ready.")

---
## 4. Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, desc):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc=desc)
    for batch in pbar:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(
            {"loss": f"{total_loss/(pbar.n+1):.4f}", "acc": f"{correct/total:.4f}"}
        )
    return total_loss / len(loader), correct / total


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

---
## 5. Training — Two-Stage Transfer Learning

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders()
criterion = nn.CrossEntropyLoss()

# ── Resume check ──
resume = os.path.exists(CHECKPOINT_PATH)
best_acc = 0.0

if resume:
    print(f"Checkpoint found at {CHECKPOINT_PATH} — loading model, skipping Stage 1")
    model = build_model().to(DEVICE)
    state = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            best_acc = json.load(f).get("best_val_acc", 0.0)
    print(f"  previous best val acc: {best_acc:.4f}")
else:
    print("No checkpoint found — starting fresh")
    model = build_model().to(DEVICE)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

### Stage 1 — Freeze backbone, train head only

In [ ]:
if not resume:
    freeze_backbone(model)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR_HEAD)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.1)

    for epoch in range(1, EPOCHS_STAGE1 + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion,
            desc=f"Stage1 Epoch {epoch}/{EPOCHS_STAGE1}",
        )
        val_loss, val_acc = validate(model, val_loader, criterion)
        scheduler.step(val_acc)
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            with open(STATE_PATH, "w") as f:
                json.dump({"best_val_acc": best_acc, "stage": 1}, f)
            print(f"  -> saved (best val acc: {best_acc:.4f})")
else:
    print("Stage 1 skipped (resuming from checkpoint)")

### Stage 2 — Unfreeze & fine-tune end-to-end

In [ ]:
unfreeze_all(model)
optimizer = torch.optim.Adam([
    {"params": [p for n, p in model.named_parameters() if "fc" not in n], "lr": LR_BACKBONE},
    {"params": model.fc.parameters(), "lr": LR_HEAD},
])
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.1)

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion,
        desc=f"Stage2 Epoch {epoch}/{EPOCHS_STAGE2}",
    )
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step(val_acc)
    print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        with open(STATE_PATH, "w") as f:
            json.dump({"best_val_acc": best_acc, "stage": 2}, f)
        print(f"  -> saved updated (best val acc: {best_acc:.4f})")

print(f"\nTraining complete. Best val acc: {best_acc:.4f}")

---
## 6. Evaluation — Metrics & Confusion Matrix

In [ ]:
def get_class_names():
    builder = load_dataset_builder("AQUA20")
    return builder.info.features["label"].names


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_top3 = [], [], []
    for batch in loader:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        outputs = model(images)
        _, preds = outputs.max(1)
        _, top3 = outputs.topk(3, dim=1)
        all_preds.append(preds.cpu())
        all_top3.append(top3.cpu())
        all_labels.append(labels.cpu())
    return (
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy(),
        torch.cat(all_top3).numpy(),
    )


def compute_topk_accuracy(labels, topk_preds, k):
    return np.mean([labels[i] in topk_preds[i, :k] for i in range(len(labels))])


# ── Run evaluation ──
class_names = get_class_names()
print(f"Classes ({len(class_names)}): {class_names}\n")

model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
)

labels, preds, top3_preds = evaluate(model, test_loader)

top1_acc = accuracy_score(labels, preds)
top3_acc = compute_topk_accuracy(labels, top3_preds, 3)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, preds, average="macro"
)

print("=" * 60)
print(f"Top-1 Accuracy:  {top1_acc:.4f} ({top1_acc * 100:.2f}%)")
print(f"Top-3 Accuracy:  {top3_acc:.4f} ({top3_acc * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}")
print("=" * 60)

print("\nPer-class Classification Report:")
print(classification_report(labels, preds, target_names=class_names, digits=4))

### Confusion Matrix

In [ ]:
matplotlib.use("Agg")
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

---
## 7. Persist Outputs

In [ ]:
for fname in [CHECKPOINT_PATH, STATE_PATH, "confusion_matrix.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(OUTPUT_DIR, fname))

print(f"Outputs saved to '{OUTPUT_DIR}/':")
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.2f} MB)")
print(f"\nTo persist across Kaggle sessions:")
print(f"  1. Download files from Data → Output tab (while kernel is running)")
print(f"  2. For resume: upload best_model.pth + training_state.json as a Dataset → mount as input")